In [64]:
import os, ast
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
import statistics

In [65]:
file = pd.read_csv('../compliance_check_updated.csv', header=0)
train = file[file['Split']!='test']
test = file[file['Split']=='test']

instruction = pd.read_csv('../../question_info/question_info.csv', header=0)
instruction = instruction.fillna('None')
instruction['choices_text'] = instruction['choices_text'].apply(ast.literal_eval)
instruction['choices_index'] = instruction['choices_index'].apply(ast.literal_eval)
instruction['addition_context'] = instruction['addition_context'].apply(ast.literal_eval)

In [66]:
def formatting(results):
    formatted_results = []
    for i in results:
        if len(i) == 1:
            formatted_results.append(int(i[0]))
        else:
            formatted_results.append([i for i, x in enumerate(i) if x == 1])
    return formatted_results
    
def turn_format(preds, truth):
    formatted_preds = []
    if type(preds[0]) == int:
        formatted_preds.extend(preds)
    else:
        pred = [0] * len(truth[0])
        pred[preds[0][0]] = 1
        formatted_preds.extend([pred]*len(truth))
    return formatted_preds

In [76]:
def get_majority(truth):
    preds = []
    if type(truth) == list:
        preds.append(list(Counter(truth).keys())[0])
    else:
        # print(list(Counter([j for i in truth for j in i]).keys())[0])
        # print(len(truth))
        preds.append([np.argmax(truth.sum().tolist())+1])
    return preds

In [80]:
for i, item in instruction.iterrows():
    if not os.path.exists('baseline/'):
        os.mkdir('baseline/')
    if item.question_index+'.csv' in os.listdir('baseline/'):
        continue
    file_item = train[['pairid', 'report_type', 'link']+[item['context']]+[col for col in file.columns if item['question_index'] in col]]
    if item['applicability'] == 'Protocol':
        file_item = file_item[file_item.report_type==1]
    elif item['applicability'] == 'Results':
        file_item = file_item[file_item.report_type==2]
    file_item = file_item.reset_index(drop=True)
    
    test_item = test[['pairid', 'report_type', 'link']+[item['context']]+[col for col in file.columns if item['question_index'] in col]]
    option_col = [col for col in file.columns if item['question_index'] in col]
    if item['type'] == 'choice':
        preds = get_majority(file_item[option_col[0]].values.tolist()) * test_item.shape[0]
    elif item['type'] == 'multilabel':
        preds = get_majority(file_item[option_col])* test_item.shape[0]
    test_item['formatted_output'] = preds
    test_item.to_csv(os.path.join('baseline/', item.question_index+'.csv'), index=False)
    
    # formatting(file_item[option_col])
    # get_majority(formatting)
    # for i, row in file_item.iterrows():
        
    #     if item['type'] == 'choice':
            
    #     elif item['type'] == 'multilabel':
        

    # truth = file[[i for i in file.columns if item in i]].dropna()
    # truth = truth.values.tolist()
    # print(truth)
    # print(o)
    # preds = formatting(truth)
    # preds = get_majority(preds)
    # preds = turn_format(preds, truth)

/var/folders/5f/g929mxqd3gl_g8qyc27lx3740000gn/T/ipykernel_52096/2698746299.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_item['formatted_output'] = preds
/var/folders/5f/g929mxqd3gl_g8qyc27lx3740000gn/T/ipykernel_52096/2698746299.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_item['formatted_output'] = preds
/var/folders/5f/g929mxqd3gl_g8qyc27lx3740000gn/T/ipykernel_52096/2698746299.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try

In [7]:
f1s_yesno = []
f1s_number = []
f1s_multilabel = []
number = ['b27a', 'b27b', 'b31a', 'p31b', 'p20b2_v2_b43efa_2', 'b31d', 'p31e']
yesno = ['b31a', 'b31d']
for f_path in all_results:
    file = pd.read_csv('gpt-4o-2024-08-06/gpt-4o-2024-08-06/'+f_path)
    item = f_path.split('.')[0]

    truth = file[[i for i in file.columns if item in i]].dropna()
    truth = truth.values.tolist()
    preds = formatting(truth)
    preds = get_majority(preds)
    preds = turn_format(preds, truth)
    if item in number:
        if item in yesno:
            f1s_yesno.append(f1_score(truth, preds, average='micro'))
        else:
            f1s_number.append(f1_score(truth, preds, average='micro'))
    else:
        f1s_multilabel.append(f1_score(truth, preds, average='micro'))

In [10]:
statistics.mean(f1s_yesno), statistics.stdev(f1s_yesno)

(0.6428571428571428, 0.10101525445522111)

In [11]:
statistics.mean(f1s_number), statistics.stdev(f1s_number)

(0.7444444444444445, 0.27666443551086073)

In [12]:
statistics.mean(f1s_multilabel), statistics.stdev(f1s_multilabel)

(0.6181818181818183, 0.33078918906655014)

In [13]:
min(f1s_yesno+f1s_number+f1s_multilabel), max(f1s_yesno+f1s_number+f1s_multilabel)

(0.23636363636363633, 1.0)

In [27]:
acc_yesno = []
acc_number = []
acc_multilabel = []
number = ['b27a', 'b27b', 'b31a', 'p31b', 'p20b2_v2_b43efa_2', 'b31d', 'p31e']
yesno = ['b31a', 'b31d']
for f_path in all_results:
    file = pd.read_csv('gpt-4o-2024-08-06/gpt-4o-2024-08-06/'+f_path)
    item = f_path.split('.')[0]

    truth = file[[i for i in file.columns if item in i]].dropna()
    truth = truth.values.tolist()
    preds = formatting(truth)
    preds = get_majority(preds)
    preds = turn_format(preds, truth)
    if item in number:
        if item in yesno:
            acc_yesno.append(accuracy_score(truth, preds))
        else:
            acc_number.append(accuracy_score(truth, preds))
    else:
        acc_multilabel.append(accuracy_score([j for i in truth for j in i], [j for i in preds for j in i]))

In [28]:
statistics.mean(acc_yesno), statistics.stdev(acc_yesno)

(0.6428571428571428, 0.10101525445522111)

In [29]:
statistics.mean(acc_number), statistics.stdev(acc_number)

(0.7444444444444445, 0.27666443551086073)

In [30]:
statistics.mean(acc_multilabel), statistics.stdev(acc_multilabel)

(0.8011111111111111, 0.23238696973734072)

In [31]:
statistics.mean(acc_yesno+acc_number+acc_multilabel), statistics.stdev(acc_yesno+acc_number+acc_multilabel)

(0.7411269841269841, 0.22473380948668192)

In [32]:
min(acc_yesno+acc_number+acc_multilabel), max(acc_yesno+acc_number+acc_multilabel)

(0.3333333333333333, 1.0)